# 03 — SHAP Interpretation & Feature Attribution

This notebook:
- Loads trained block classifiers and their SHAP values
- Summarises global and per-class feature importance
- Shows which feature blocks contribute most (block attribution)
- Explores attention weights from the deep fusion model
- Produces publication-quality figures for key findings

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path

from fungal_classifier.utils.io import load_metadata, load_feature_blocks
from fungal_classifier.models.block_classifier import BlockClassifier
from fungal_classifier.models.fusion_model import StackingFusionModel
from fungal_classifier.evaluation.shap_analysis import (
    compute_shap_values,
    mean_absolute_shap,
    per_class_shap_summary,
    block_level_importance,
    plot_shap_summary,
    plot_block_contributions,
    plot_per_class_heatmap,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})
sns.set_style("whitegrid")
shap.initjs()


## 1. Load models and data

In [ ]:
METADATA_PATH = Path("../data/raw/metadata.tsv")
FEATURES_DIR  = Path("../data/features/")
MODEL_DIR     = Path("../results/")  # Update to your results directory
TARGET        = "taxonomy_order"     # Update to your target

metadata = load_metadata(METADATA_PATH)
blocks = load_feature_blocks(FEATURES_DIR)

# Load block classifiers
block_classifiers = {}
for pkl in (MODEL_DIR / "models").glob("block_*.pkl"):
    name = pkl.stem.replace("block_", "")
    block_classifiers[name] = BlockClassifier.load(pkl)
    print(f"Loaded classifier: {name}")

fusion_model = StackingFusionModel.load(MODEL_DIR / "models" / "fusion_model.pkl")
class_names  = list(fusion_model.classes_)
print(f"\nClasses ({len(class_names)}): {class_names[:5]}{'...' if len(class_names) > 5 else ''}")


## 2. Global feature importance — per block

In [ ]:
# Load or compute SHAP values
shap_dir = MODEL_DIR / "shap"
all_mean_abs = {}

for block_name, clf in block_classifiers.items():
    if block_name not in blocks:
        continue
    X = blocks[block_name]
    # Try loading pre-computed
    shap_csv = shap_dir / f"{block_name}_mean_abs_shap.csv"
    if shap_csv.exists():
        mean_abs = pd.read_csv(shap_csv, index_col=0).squeeze()
        print(f"Loaded SHAP for {block_name} from cache")
    else:
        print(f"Computing SHAP for {block_name}...")
        shap_values = compute_shap_values(clf, X)
        mean_abs = mean_absolute_shap(shap_values, X.columns.tolist())
        shap_dir.mkdir(exist_ok=True)
        mean_abs.to_csv(shap_csv)
    all_mean_abs[block_name] = mean_abs


In [ ]:
# Top 20 features per block
fig, axes = plt.subplots(1, len(all_mean_abs), figsize=(5 * len(all_mean_abs), 5))
if len(all_mean_abs) == 1:
    axes = [axes]

colors = ["#1a6fa8", "#e8722a", "#2ca02c", "#d62728", "#9467bd"]
for ax, ((name, mean_abs), color) in zip(axes, zip(all_mean_abs.items(), colors * 5)):
    top20 = mean_abs.head(20).sort_values()
    top20.plot(kind="barh", ax=ax, color=color, edgecolor="white")
    ax.set_xlabel("Mean |SHAP|")
    ax.set_title(f"{name}\nTop 20 features")
    ax.tick_params(labelsize=8)

plt.suptitle(f"Global SHAP Importance per Block — {TARGET}", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


## 3. Block-level attribution

Which feature type explains the most variance in predictions?

In [ ]:
# For concat fusion: aggregate feature importances by block prefix
if (MODEL_DIR / "features_fused.parquet").exists():
    X_fused = pd.read_parquet(MODEL_DIR / "features_fused.parquet")
    fusion_clf = block_classifiers.get("fusion")
    if fusion_clf:
        fused_shap = compute_shap_values(fusion_clf, X_fused)
        fused_mean_abs = mean_absolute_shap(fused_shap, X_fused.columns.tolist())
        block_attr = block_level_importance(fused_mean_abs)
        fig = plot_block_contributions(block_attr,
                                        title=f"Feature Block Contributions — {TARGET}")
        plt.show()
else:
    # Aggregate from per-block mean |SHAP| (proxy)
    block_totals = {name: float(ms.sum()) for name, ms in all_mean_abs.items()}
    block_attr = pd.Series(block_totals).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = plt.cm.tab10(np.linspace(0, 1, len(block_attr)))
    block_attr.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
    ax.set_xlabel("Total Mean |SHAP| (sum across top features)")
    ax.set_title(f"Feature Block Attribution — {TARGET}")
    plt.tight_layout()
    plt.show()

print("Block attribution:")
for block, val in block_attr.sort_values(ascending=False).items():
    print(f"  {block:20s}  {val:.4f}  ({val/block_attr.sum():.1%})")


## 4. Per-class SHAP heatmap

Which features distinguish each class?

In [ ]:
# Use domain block as example (typically most informative)
FOCUS_BLOCK = "domains"
if FOCUS_BLOCK in block_classifiers and FOCUS_BLOCK in blocks:
    clf = block_classifiers[FOCUS_BLOCK]
    X   = blocks[FOCUS_BLOCK]

    shap_values = compute_shap_values(clf, X)
    class_summary = per_class_shap_summary(
        shap_values, X.columns.tolist(), class_names, top_n=15
    )

    fig = plot_per_class_heatmap(class_summary, top_n=15)
    plt.title(f"SHAP per Class — {FOCUS_BLOCK} block")
    plt.show()

    # Top 5 features per class
    print(f"\nTop 5 discriminating features per class ({FOCUS_BLOCK}):")
    for cls in class_names[:8]:
        top5 = class_summary[class_summary["class"] == cls].nsmallest(5, "rank")["feature"].tolist()
        print(f"  {cls:30s}: {', '.join(top5)}")


## 5. Deep fusion attention weights

If the deep model was trained, visualise which blocks it attended to per genome.

In [ ]:
attention_path = MODEL_DIR / "attention_weights.csv"
if attention_path.exists():
    attn_df = pd.read_csv(attention_path, index_col="genome_id")
    print(f"Attention weights loaded: {attn_df.shape}")

    # Mean attention per block
    mean_attn = attn_df.mean(axis=0).sort_values(ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    mean_attn.plot(kind="bar", ax=axes[0], color="#1a6fa8", edgecolor="white")
    axes[0].set_ylabel("Mean attention weight")
    axes[0].set_title("Mean block attention across all genomes")
    axes[0].tick_params(axis="x", rotation=30)

    # Per-class attention heatmap
    if TARGET in metadata.columns:
        meta_common = metadata.loc[attn_df.index.intersection(metadata.index), TARGET]
        attn_by_class = attn_df.join(meta_common).groupby(TARGET).mean()
        sns.heatmap(attn_by_class.T, ax=axes[1], cmap="YlOrRd",
                    linewidths=0.5, annot=True, fmt=".2f", annot_kws={"size": 8})
        axes[1].set_title(f"Mean attention by {TARGET}")
        axes[1].set_xlabel(TARGET)

    plt.suptitle("Deep Fusion Block Attention Weights", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"Attention weights not found at {attention_path}.")
    print("Train the deep fusion model with --model-type deep and --config configs/deep_fusion.yaml")


## 6. SHAP force plot for individual genomes

Inspect why the model made a specific prediction.

In [ ]:
GENOME_OF_INTEREST = genome_ids[0]  # Replace with a specific genome ID
FOCUS_BLOCK = "domains"

if FOCUS_BLOCK in block_classifiers and FOCUS_BLOCK in blocks:
    clf = block_classifiers[FOCUS_BLOCK]
    X   = blocks[FOCUS_BLOCK]

    if GENOME_OF_INTEREST in X.index:
        model = clf._model
        explainer = shap.TreeExplainer(model)
        sample = X.loc[[GENOME_OF_INTEREST]].values
        sv = explainer.shap_values(sample)

        # For multiclass, show force plot for the predicted class
        y_pred_enc = model.predict(sample)[0]
        if isinstance(sv, list):
            sv_for_class = sv[y_pred_enc]
        else:
            sv_for_class = sv

        pred_class = clf._label_encoder.inverse_transform([y_pred_enc])[0]
        print(f"Genome: {GENOME_OF_INTEREST}")
        print(f"Predicted class: {pred_class}")

        shap.force_plot(
            explainer.expected_value[y_pred_enc] if isinstance(explainer.expected_value, list)
            else explainer.expected_value,
            sv_for_class[0],
            X.loc[GENOME_OF_INTEREST],
            matplotlib=True,
            show=False,
        )
        plt.title(f"SHAP force plot — {GENOME_OF_INTEREST} — predicted: {pred_class}")
        plt.tight_layout()
        plt.show()
    else:
        print(f"Genome {GENOME_OF_INTEREST} not found in {FOCUS_BLOCK} block.")


## 7. Export figures for publication

In [ ]:
FIG_DIR = Path("../results/figures/")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Block contribution bar chart
if 'block_attr' in dir():
    fig, ax = plt.subplots(figsize=(7, 4))
    block_attr.sort_values().plot(kind="barh", ax=ax, color="#1a6fa8", edgecolor="white")
    ax.set_xlabel("Total Mean |SHAP| contribution")
    ax.set_title(f"Feature Block Attribution — {TARGET}")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "block_attribution.svg", dpi=150, bbox_inches="tight")
    print(f"Saved: {FIG_DIR / 'block_attribution.svg'}")

print("Figures exported.")
